# 01. Data Augmentation

원본 die-level 데이터 (174,980 rows) 를 시드 삼아 합성 dies 를 생성하고 DuckDB 의 `x_data` / `y_data` 테이블로 적재한다.

**구조**
- `x_data`: die level (ufs_serial, run_wf_xy, position, X0~X1086)
- `y_data`: unit level (ufs_serial, health) — 4 dies 가 1 unit 으로 health 공유
- 매핑키 `ufs_serial` 9 자리 zero-pad (e.g., `S000000000`)
- 원본 174,980 rows 가 먼저, 그 아래로 합성 rows

**Y 샘플링**
- P(y > 0.5) = 1/300,000
- P(0.2 < y <= 0.5) = 1/400,000  (원본 0개)
- P(0.1 < y <= 0.2) = 1/250,000  (원본 1개)
- 나머지 y in [0, 0.1] : 원본 empirical (zero spike 70.8% 자연 보존)

**X 매칭** (per die)
1. (die_x, die_y) +/- XY_TOL 셀 이웃 후보 풀
2. 후보 중 health 정렬 후 target_y 기준 위 5 개 + 아래 5 개 (총 10)
3. 10 개의 (max - min) > Y_SPREAD_TRIGGER 면 위 3 + 아래 3 (총 6) 으로 축소
4. 그 중 1개 random pick 후 X0~X1086 통째로 복사

**Wafer / Lot**
- 1 lot = 25 wafers, run_id 7-digit, 원본 max 다음부터 증분
- 각 wafer yield (dies 수) 는 원본 분포에서 샘플 (mean ~ 405)
- 위치 sort by (die_y, die_x) 후 4 dies 씩 unit 으로 그룹 (자연 same-row 패턴)

## 1. Parameters (모든 knob 상단 노출)

In [2]:
# Generation
N_TARGET_ROWS = 10_000_000   # test target; full target = 700_000_000
SEED          = 42
CHUNK_ROWS    = 100_000      # rows per DuckDB insert chunk (tuned for ~1GB peak)

# Y distribution probabilities
P_GT_05 = 1 / 300_000
P_02_05 = 1 / 400_000
P_01_02 = 1 / 250_000

# Feature matching
XY_TOL            = 1     # die_x, die_y +/- XY_TOL
K_NEIGHBORS_BIG   = 5     # 5 above + 5 below = 10
K_NEIGHBORS_SMALL = 3     # 3 above + 3 below = 6 (when spread too large)
Y_SPREAD_TRIGGER  = 0.05  # if max-min of 10 > this, shrink to 6

# Wafer / lot structure
WAFERS_PER_LOT = 25
DIES_PER_UNIT  = 4

# Format
UFS_DIGITS    = 9   # S000000000
RUN_ID_DIGITS = 7   # 0000000

# Date placeholder (나중에 일괄 수정)
DATE_START = 20201107
DATE_END   = 20210506

## 2. Imports + paths

In [5]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from tqdm.auto import tqdm

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / '0_data').exists() and (cand / '5_dashboard').exists():
            return cand
    raise RuntimeError('project root not found')

PROJECT_ROOT    = find_project_root()
SOURCE_DATA_DIR = PROJECT_ROOT / '0_data'
DASHBOARD_DIR   = PROJECT_ROOT / '5_dashboard'
OUTPUT_DIR      = DASHBOARD_DIR / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_PATH     = OUTPUT_DIR / 'augmented.duckdb'

rng = np.random.default_rng(SEED)
print('source :', SOURCE_DATA_DIR)
print('output :', DUCKDB_PATH)

source : C:\Users\Dell5371\Desktop\기업연계프로젝트\0_data
output : C:\Users\Dell5371\Desktop\기업연계프로젝트\5_dashboard\data\augmented.duckdb


## 3. Load original + parse run_wf_xy + attach health

In [6]:
xs = pd.read_csv(SOURCE_DATA_DIR / 'compet_xs_data.csv').drop(columns=['split'], errors='ignore')
ys = pd.concat([
    pd.read_csv(SOURCE_DATA_DIR / 'compet_ys_train_data.csv'),
    pd.read_csv(SOURCE_DATA_DIR / 'compet_ys_validation_data.csv'),
    pd.read_csv(SOURCE_DATA_DIR / 'compet_ys_test_data.csv'),
], ignore_index=True)
print(f'xs: {xs.shape}, ys: {ys.shape}')

# parse run_wf_xy = run_id _ wafer_no _ die_x _ die_y
parts = xs['run_wf_xy'].str.split('_', expand=True)
xs['run_id_int'] = parts[0].astype(int)
xs['wafer_no']   = parts[1].astype(int)
xs['die_x']      = parts[2].astype(int)
xs['die_y']      = parts[3].astype(int)

# attach die-level health (4 dies of one unit share unit health)
xs = xs.merge(ys[['ufs_serial', 'health']], on='ufs_serial', how='left')
print(f'xs+health: {xs.shape}, missing health: {xs["health"].isna().sum()}')

xs: (174980, 1090), ys: (43745, 2)
xs+health: (174980, 1095), missing health: 0


## 4. Build feature matching index

각 (die_x, die_y) 셀에 대해 +/- XY_TOL 이웃을 모아 health 정렬한 (indices, healths) 튜플 사전. 매칭 시 binary search 로 사용.

In [7]:
# --- extract numpy arrays so xs can be freed later ---
feat_cols = [c for c in xs.columns if c.startswith('X') and c[1:].isdigit()]
print(f'feat_cols: {len(feat_cols)} (X0..X{len(feat_cols)-1})')

X_orig          = xs[feat_cols].astype(np.float32).values
health_orig     = xs['health'].values.astype(np.float32)
die_x_orig      = xs['die_x'].values.astype(np.int32)
die_y_orig      = xs['die_y'].values.astype(np.int32)
orig_ufs_str    = xs['ufs_serial'].values.copy()                  # 'S00000' format
orig_run_id_int = xs['run_id_int'].values.astype(np.int32)
orig_wafer_no   = xs['wafer_no'].values.astype(np.int8)           # max 25
orig_position   = xs['position'].values.astype(np.int8)
print(f'X_orig: {X_orig.shape}  mem={X_orig.nbytes/1e6:.0f} MB  dtype={X_orig.dtype}')

# --- cell -> die indices (vectorized via groupby.indices) ---
cell_to_indices = {
    k: v.astype(np.int64)
    for k, v in pd.DataFrame({'die_x': die_x_orig, 'die_y': die_y_orig})
                  .groupby(['die_x', 'die_y']).indices.items()
}
print(f'unique cells: {len(cell_to_indices)}')

# --- per cell neighborhood (+/-XY_TOL) sorted by health ---
nbr_sorted_idx    = {}
nbr_sorted_health = {}
for (cx, cy) in cell_to_indices:
    pool_lists = []
    for dx in range(-XY_TOL, XY_TOL + 1):
        for dy in range(-XY_TOL, XY_TOL + 1):
            arr = cell_to_indices.get((cx + dx, cy + dy))
            if arr is not None:
                pool_lists.append(arr)
    pool_arr = np.concatenate(pool_lists)
    h        = health_orig[pool_arr]
    order    = np.argsort(h)                                      # default quicksort
    nbr_sorted_idx[(cx, cy)]    = pool_arr[order]
    nbr_sorted_health[(cx, cy)] = h[order]

print(f'cells with neighborhood: {len(nbr_sorted_idx)}')
print(f'avg pool size: {np.mean([len(v) for v in nbr_sorted_idx.values()]):.0f}')

feat_cols: 1087 (X0..X1086)
X_orig: (174980, 1087)  mem=761 MB  dtype=float32
unique cells: 953
cells with neighborhood: 953
avg pool size: 1612


## 5. Sample distributions (Y empirical, yield, cell freq, layout ratios)

In [8]:
# Y empirical for [0, 0.1] (preserves zero spike ~70.8%)
ys_low = ys[ys['health'] <= 0.1]['health'].values.astype(np.float32)
print(f'ys_low: {len(ys_low):,}  zero_pct={(ys_low==0).mean()*100:.1f}%')

# yield (dies per wafer) -- compute from xs while it's alive
yield_dist = xs.groupby(['run_id_int', 'wafer_no']).size().values
print(f'yield: min={yield_dist.min()}, max={yield_dist.max()}, mean={yield_dist.mean():.0f}, median={np.median(yield_dist):.0f}')

# cell frequency (for weighted position sampling)
all_cells_list = list(cell_to_indices.keys())
cell_freq      = np.array([len(cell_to_indices[c]) for c in all_cells_list], dtype=np.float64)
cell_probs     = cell_freq / cell_freq.sum()
all_cells_arr  = np.array(all_cells_list, dtype=np.int32)  # shape (n_cells, 2)
print(f'all_cells_arr: {all_cells_arr.shape}')

# original counts / IDs
ORIG_MAX_RUN_ID  = int(orig_run_id_int.max())
ORIG_N_UNITS     = int(ys['ufs_serial'].nunique())
ORIG_N_DIES      = int(len(orig_ufs_str))
orig_ufs_int     = np.array([int(s[1:]) for s in ys['ufs_serial'].values], dtype=np.int64)
ORIG_MAX_UFS_INT = int(orig_ufs_int.max())
print(f'orig max run_id: {ORIG_MAX_RUN_ID}, orig units: {ORIG_N_UNITS}, orig dies: {ORIG_N_DIES}')
print(f'orig max ufs_int: {ORIG_MAX_UFS_INT}')

# free xs (no longer needed; metadata extracted to numpy in build-index, yield_dist captured above)
del xs
print('xs freed')

ys_low: 43,742  zero_pct=70.8%
yield: min=17, max=516, mean=405, median=424
all_cells_arr: (953, 2)
orig max run_id: 27, orig units: 43745, orig dies: 174980
orig max ufs_int: 43752
xs freed


## 6. Y sampling function

In [9]:
def sample_y_values(n: int, rng) -> np.ndarray:
    """Sample n unit-level health values per spec."""
    u = rng.random(n)
    y = np.empty(n, dtype=np.float32)
    cum1 = P_GT_05
    cum2 = cum1 + P_02_05
    cum3 = cum2 + P_01_02
    m_gt05 = u < cum1
    m_25   = (u >= cum1) & (u < cum2)
    m_12   = (u >= cum2) & (u < cum3)
    m_low  = u >= cum3
    n_gt05, n_25, n_12, n_low = m_gt05.sum(), m_25.sum(), m_12.sum(), m_low.sum()
    if n_gt05: y[m_gt05] = rng.uniform(0.5, 1.0, n_gt05)
    if n_25:   y[m_25]   = rng.uniform(0.2, 0.5, n_25)
    if n_12:   y[m_12]   = rng.uniform(0.1, 0.2, n_12)
    if n_low:  y[m_low]  = rng.choice(ys_low, n_low, replace=True)
    return y

# small sanity check (100k samples)
_ts = sample_y_values(100_000, rng)
print(f'sample_y test (100k): zero={(_ts==0).mean()*100:.2f}%, mean={_ts.mean():.6f}, max={_ts.max():.4f}')
print(f'  >0.5: {(_ts>0.5).sum()}, 0.2-0.5: {((_ts>0.2)&(_ts<=0.5)).sum()}, 0.1-0.2: {((_ts>0.1)&(_ts<=0.2)).sum()}')

sample_y test (100k): zero=70.78%, mean=0.002491, max=0.0974
  >0.5: 0, 0.2-0.5: 0, 0.1-0.2: 0


## 7. Wafer / unit / position generation

각 wafer 마다 yield 만큼 cell 을 무복원 가중샘플 → (die_y, die_x) 로 정렬 → 4 개씩 unit. 자연스럽게 same-row 가 주류가 되며 row 경계에서 2-row 가 발생 (원본 83/12/5 와 근사).

In [10]:
def generate_wafer_cells(yield_size: int, rng):
    """Sample yield_size unique cells (rounded down to multiple of 4) for one wafer, sorted by (die_y, die_x)."""
    n = (yield_size // DIES_PER_UNIT) * DIES_PER_UNIT
    n = max(DIES_PER_UNIT, min(n, len(all_cells_arr)))
    idx = rng.choice(len(all_cells_arr), size=n, replace=False, p=cell_probs)
    cells = all_cells_arr[idx]                                           # (n, 2)
    order = np.lexsort((cells[:, 0], cells[:, 1]))                       # primary die_y, secondary die_x
    return cells[order]                                                  # ascending (die_y, die_x)

# quick test
_test_cells = generate_wafer_cells(404, rng)
print(f'test wafer: {_test_cells.shape}, first 8 cells:\n{_test_cells[:8]}')

test wafer: (404, 2), first 8 cells:
[[37 11]
 [38 11]
 [40 11]
 [41 11]
 [44 11]
 [47 11]
 [31 12]
 [34 12]]


## 8. Generate metadata for all synthetic dies

wafer-by-wafer 로 yield 샘플 + cell 샘플 + unit 그룹핑 + position 1~4 부여 + y 샘플. 결과는 큰 numpy array 들 (메타데이터만, feature 는 다음 단계에서 매칭).

In [11]:
MEAN_YIELD = float(yield_dist.mean())
n_wafers_target = int(np.ceil(N_TARGET_ROWS / MEAN_YIELD * 1.05))
n_lots_target   = int(np.ceil(n_wafers_target / WAFERS_PER_LOT))
print(f'target rows: {N_TARGET_ROWS:,}')
print(f'wafers est : {n_wafers_target:,}  (lots {n_lots_target:,})')

# pre-allocate over-sized buffers; trim to N_TARGET_ROWS at end
buf_size = int(N_TARGET_ROWS * 1.05)
die_x_synth    = np.empty(buf_size, dtype=np.int32)
die_y_synth    = np.empty(buf_size, dtype=np.int32)
y_synth        = np.empty(buf_size, dtype=np.float32)
wafer_no_synth = np.empty(buf_size, dtype=np.int8)     # max 25
run_id_synth   = np.empty(buf_size, dtype=np.int32)    # large pool for 7억 scale
position_synth = np.empty(buf_size, dtype=np.int8)
unit_id_synth  = np.empty(buf_size, dtype=np.int32)    # max ~175M, fits int32

t0 = time.time()
row = 0
unit_counter = 0
next_run_id  = ORIG_MAX_RUN_ID + 1
wafer_in_lot = 0
current_run_id = next_run_id

wafer_yields = rng.choice(yield_dist, size=n_wafers_target, replace=True)

pbar = tqdm(range(n_wafers_target), desc='wafers', mininterval=0.5)
for wi in pbar:
    if row >= N_TARGET_ROWS:
        break
    if wafer_in_lot == WAFERS_PER_LOT:
        next_run_id   += 1
        current_run_id = next_run_id
        wafer_in_lot   = 0
    wafer_no = wafer_in_lot + 1
    wafer_in_lot += 1

    cells = generate_wafer_cells(int(wafer_yields[wi]), rng)
    n_cells = len(cells)
    n_units = n_cells // DIES_PER_UNIT
    if n_units == 0:
        continue

    pos_perms = rng.permuted(
        np.tile(np.arange(1, DIES_PER_UNIT + 1, dtype=np.int8), n_units).reshape(n_units, DIES_PER_UNIT),
        axis=1,
    ).ravel()
    y_units  = sample_y_values(n_units, rng)
    y_repeat = np.repeat(y_units, DIES_PER_UNIT)
    unit_ids = np.repeat(np.arange(unit_counter, unit_counter + n_units, dtype=np.int32), DIES_PER_UNIT)

    end = row + n_cells
    if end > buf_size:
        n_cells = buf_size - row
        n_cells -= n_cells % DIES_PER_UNIT
        if n_cells <= 0:
            break
        cells, pos_perms, y_repeat, unit_ids = cells[:n_cells], pos_perms[:n_cells], y_repeat[:n_cells], unit_ids[:n_cells]
        end = row + n_cells

    die_x_synth   [row:end] = cells[:, 0]
    die_y_synth   [row:end] = cells[:, 1]
    y_synth       [row:end] = y_repeat
    wafer_no_synth[row:end] = wafer_no
    run_id_synth  [row:end] = current_run_id
    position_synth[row:end] = pos_perms
    unit_id_synth [row:end] = unit_ids
    row          += n_cells
    unit_counter += (n_cells // DIES_PER_UNIT)
pbar.close()

# truncate to exactly N_TARGET_ROWS (drop remainder mod 4 if any)
row_final = min(row, N_TARGET_ROWS)
row_final -= row_final % DIES_PER_UNIT
die_x_synth    = die_x_synth   [:row_final]
die_y_synth    = die_y_synth   [:row_final]
y_synth        = y_synth       [:row_final]
wafer_no_synth = wafer_no_synth[:row_final]
run_id_synth   = run_id_synth  [:row_final]
position_synth = position_synth[:row_final]
unit_id_synth  = unit_id_synth [:row_final]
print(f'metadata done: rows={row_final:,}  units={row_final//DIES_PER_UNIT:,}  lots={(next_run_id - ORIG_MAX_RUN_ID):,}  elapsed={time.time()-t0:.1f}s')

target rows: 10,000,000
wafers est : 25,923  (lots 1,037)


wafers:   0%|          | 0/25923 [00:00<?, ?it/s]

metadata done: rows=10,000,000  units=2,500,000  lots=990  elapsed=4.9s


## 9. Feature matching (per-cell vectorized)

각 cell 그룹에 대해 vectorized 처리: target_y 의 binary search 위치 → +/-5 후보 → spread 검사 → 필요 시 +/-3 으로 축소 → random offset 으로 1개 선택. 결과는 원본 die index array (length = row_final).

In [12]:
t0 = time.time()
picked_idx = np.empty(row_final, dtype=np.int64)

# group synthetic rows by (die_x, die_y) cell
cell_key = die_x_synth.astype(np.int64) * 100_000 + die_y_synth.astype(np.int64)
order = np.argsort(cell_key)                                              # default quicksort
sorted_keys = cell_key[order]
boundaries = np.concatenate([[0], np.where(np.diff(sorted_keys) != 0)[0] + 1, [len(sorted_keys)]])
n_groups = len(boundaries) - 1

pbar = tqdm(range(n_groups), desc='cell groups', mininterval=0.3)
for g in pbar:
    s, e   = boundaries[g], boundaries[g+1]
    rowsel = order[s:e]
    cx, cy = int(die_x_synth[rowsel[0]]), int(die_y_synth[rowsel[0]])
    nbr    = nbr_sorted_idx[(cx, cy)]                                     # synth cells always in index
    sh     = nbr_sorted_health[(cx, cy)]
    ty     = y_synth[rowsel]
    n_pool = len(sh)
    insert = np.searchsorted(sh, ty)

    # full +/- BIG range for spread test
    lo_b = np.maximum(insert - K_NEIGHBORS_BIG, 0)
    hi_b = np.minimum(insert + K_NEIGHBORS_BIG - 1, n_pool - 1)
    spread = sh[hi_b] - sh[lo_b]
    shrink = spread > Y_SPREAD_TRIGGER

    K = np.where(shrink, K_NEIGHBORS_SMALL, K_NEIGHBORS_BIG).astype(np.int64)
    lo = np.maximum(insert - K, 0)
    hi = np.minimum(insert + K - 1, n_pool - 1)
    rng_len = (hi - lo + 1).clip(min=1)
    off = (rng.random(len(rowsel)) * rng_len).astype(np.int64)
    off = np.minimum(off, rng_len - 1)
    sel_pos = lo + off
    picked_idx[rowsel] = nbr[sel_pos]
pbar.close()

print(f'matched {row_final:,} rows in {time.time()-t0:.1f}s')

cell groups:   0%|          | 0/953 [00:00<?, ?it/s]

matched 10,000,000 rows in 1.2s


## 10. Insert original 174,980 dies into DuckDB

원본 ufs_serial 5 자리 -> 9 자리 zero-pad 로 통일. run_id 7 자리 zero-pad.

In [13]:
if DUCKDB_PATH.exists():
    DUCKDB_PATH.unlink()
    print('removed existing DuckDB file')
con = duckdb.connect(str(DUCKDB_PATH))

# ----- y_data: original units (single shot, 43k rows) -----
ys_out = pd.DataFrame({
    'ufs_serial': [f'S{int(i):0{UFS_DIGITS}d}' for i in orig_ufs_int],
    'health'    : ys['health'].values.astype(np.float32),
})
con.execute('CREATE TABLE y_data AS SELECT * FROM ys_out')
del ys_out
print(f'y_data orig inserted: {ORIG_N_UNITS:,}')

# ----- x_data: original dies, chunked (lower memory peak) -----
# pre-build padded id strings once (174k strings, ~10 MB)
xs_ufs_int    = np.array([int(s[1:]) for s in orig_ufs_str], dtype=np.int64)
xs_ufs_padded = [f'S{int(i):0{UFS_DIGITS}d}' for i in xs_ufs_int]
xs_run_wf_xy  = [
    f'{int(r):0{RUN_ID_DIGITS}d}_{int(w)}_{int(x)}_{int(y)}'
    for r, w, x, y in zip(orig_run_id_int, orig_wafer_no, die_x_orig, die_y_orig)
]

table_created = False
n_chunks_orig = (ORIG_N_DIES + CHUNK_ROWS - 1) // CHUNK_ROWS
pbar = tqdm(range(0, ORIG_N_DIES, CHUNK_ROWS), desc='x_data orig', total=n_chunks_orig, mininterval=0.5)
for cstart in pbar:
    cend = min(cstart + CHUNK_ROWS, ORIG_N_DIES)
    sl = slice(cstart, cend)
    chunk_df = pd.DataFrame({
        'ufs_serial': xs_ufs_padded[cstart:cend],
        'run_wf_xy' : xs_run_wf_xy[cstart:cend],
        'position'  : orig_position[sl],
    })
    chunk_df = pd.concat([chunk_df.reset_index(drop=True), pd.DataFrame(X_orig[sl], columns=feat_cols)], axis=1)
    if not table_created:
        con.execute('CREATE TABLE x_data AS SELECT * FROM chunk_df')
        table_created = True
    else:
        con.execute('INSERT INTO x_data SELECT * FROM chunk_df')
    del chunk_df
pbar.close()
print(f'x_data orig inserted: {ORIG_N_DIES:,}')

y_data orig inserted: 43,745


x_data orig:   0%|          | 0/2 [00:00<?, ?it/s]

x_data orig inserted: 174,980


## 11. Insert synthetic chunks into DuckDB

메모리 부담을 피하기 위해 CHUNK_ROWS rows 씩 잘라서 INSERT. ufs_serial 은 ORIG_N_UNITS 부터 unit_id_synth 만큼 증분.

In [14]:
t0 = time.time()
# ----- y_data: synth units (every 4th die = 1 unit) -----
unit_unique_idx = np.arange(0, row_final, DIES_PER_UNIT)
unit_y          = y_synth[unit_unique_idx]
unit_ids_seq    = unit_id_synth[unit_unique_idx].astype(np.int64)
ufs_int_synth   = ORIG_MAX_UFS_INT + 1 + unit_ids_seq
ufs_synth_unit  = [f'S{int(i):0{UFS_DIGITS}d}' for i in ufs_int_synth]

ys_synth_df = pd.DataFrame({'ufs_serial': ufs_synth_unit, 'health': unit_y})
con.execute('INSERT INTO y_data SELECT * FROM ys_synth_df')
del ys_synth_df, ufs_synth_unit
print(f'y_data synth inserted: {len(unit_unique_idx):,}  (elapsed {time.time()-t0:.1f}s)')

# ----- x_data: synth dies, chunked -----
ufs_int_die = ORIG_MAX_UFS_INT + 1 + unit_id_synth.astype(np.int64)   # length row_final

n_chunks = (row_final + CHUNK_ROWS - 1) // CHUNK_ROWS
pbar = tqdm(range(0, row_final, CHUNK_ROWS), desc='x_data synth', total=n_chunks, mininterval=0.5)
for cstart in pbar:
    cend = min(cstart + CHUNK_ROWS, row_final)
    sl   = slice(cstart, cend)
    ufs_chunk  = [f'S{int(i):0{UFS_DIGITS}d}' for i in ufs_int_die[sl]]
    rwxy_chunk = [
        f'{int(r):0{RUN_ID_DIGITS}d}_{int(w)}_{int(x)}_{int(y)}'
        for r, w, x, y in zip(run_id_synth[sl], wafer_no_synth[sl], die_x_synth[sl], die_y_synth[sl])
    ]
    feat_chunk = X_orig[picked_idx[sl]]
    chunk_df = pd.DataFrame({
        'ufs_serial': ufs_chunk,
        'run_wf_xy' : rwxy_chunk,
        'position'  : position_synth[sl],
    })
    chunk_df = pd.concat([chunk_df.reset_index(drop=True), pd.DataFrame(feat_chunk, columns=feat_cols)], axis=1)
    con.execute('INSERT INTO x_data SELECT * FROM chunk_df')
    del chunk_df, feat_chunk
pbar.close()

print(f'x_data synth inserted: {row_final:,}  total elapsed {time.time()-t0:.1f}s')

y_data synth inserted: 2,500,000  (elapsed 1.9s)


x_data synth:   0%|          | 0/100 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

x_data synth inserted: 10,000,000  total elapsed 1072.1s


## 12. Verify counts + sample

In [15]:
n_x = con.execute('SELECT COUNT(*) FROM x_data').fetchone()[0]
n_y = con.execute('SELECT COUNT(*) FROM y_data').fetchone()[0]
print(f'x_data: {n_x:,}  (orig {ORIG_N_DIES:,} + synth {n_x-ORIG_N_DIES:,})')
print(f'y_data: {n_y:,}  (orig {ORIG_N_UNITS:,} + synth {n_y-ORIG_N_UNITS:,})')

print()
print('=== x_data sample ===')
print(con.execute('SELECT ufs_serial, run_wf_xy, position, X0, X1, X1086 FROM x_data LIMIT 3').fetchdf())
print(con.execute('SELECT ufs_serial, run_wf_xy, position, X0, X1, X1086 FROM x_data ORDER BY ufs_serial DESC LIMIT 3').fetchdf())

print()
print('=== y_data sample ===')
print(con.execute('SELECT * FROM y_data LIMIT 3').fetchdf())
print(con.execute('SELECT * FROM y_data ORDER BY ufs_serial DESC LIMIT 3').fetchdf())

print()
print('=== synth Y bucket counts ===')
print(con.execute('''
SELECT
  COUNT(*) FILTER (WHERE health = 0)              AS zero_cnt,
  COUNT(*) FILTER (WHERE health > 0 AND health <= 0.1)   AS le_01,
  COUNT(*) FILTER (WHERE health > 0.1 AND health <= 0.2) AS bw_01_02,
  COUNT(*) FILTER (WHERE health > 0.2 AND health <= 0.5) AS bw_02_05,
  COUNT(*) FILTER (WHERE health > 0.5)            AS gt_05
FROM y_data
''').fetchdf())

con.close()
print()
print(f'DuckDB file: {DUCKDB_PATH}  size={DUCKDB_PATH.stat().st_size/(1024**3):.2f} GB')

x_data: 10,174,980  (orig 174,980 + synth 10,000,000)
y_data: 2,543,745  (orig 43,745 + synth 2,500,000)

=== x_data sample ===
   ufs_serial         run_wf_xy  position    X0     X1       X1086
0  S000000000  0000000_25_24_25         1  0.04  0.040  20201112.0
1  S000000001  0000000_10_41_30         1  0.92  0.985  20201112.0
2  S000000002   0000000_8_37_24         1  0.02    NaN  20201112.0
   ufs_serial         run_wf_xy  position    X0    X1       X1086
0  S002543752  0001017_22_47_24         4  0.04  0.04  20201108.0
1  S002543752  0001017_22_51_24         1  0.02  0.02  20201108.0
2  S002543752  0001017_22_53_24         3  0.02  0.02  20201108.0

=== y_data sample ===
   ufs_serial  health
0  S000000000     0.0
1  S000000002     0.0
2  S000000003     0.0
   ufs_serial   health
0  S002543752  0.00000
1  S002543751  0.00352
2  S002543750  0.00000

=== synth Y bucket counts ===
   zero_cnt   le_01  bw_01_02  bw_02_05  gt_05
0   1802132  741587         7         7     12

DuckDB file